In [ ]:
# ============================================================
# ENSEMBLE TESTING WITH GROUND TRUTH
# ============================================================

!pip install ultralytics pandas --quiet

from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
import os
from ultralytics import YOLO
from collections import Counter
import pandas as pd

# Load models
print("Loading models...")
v21 = YOLO("/content/drive/MyDrive/TeamProject/models/srilankan_food_model_v21_74.5.pt")
v24 = YOLO("/content/drive/MyDrive/TeamProject/models/srilankan_food_model_v24_71.9.pt")
v25 = YOLO("/content/drive/MyDrive/TeamProject/models/srilankan_food_model_v25_70.5.pt")
print("✅ Models loaded!")

# ============================================================
# GROUND TRUTH - UPDATE WITH YOUR ACTUAL FOODS
# ============================================================

ground_truth = {
    'test1.jpg': ['kottu'],
    'test2.jpg': ['kiribath', 'lunu sambol'],
    'test3.jpg': ['cocount roti', 'chicken curry', 'lunu sambol'],
    'test4.jpg': ['fried rice'],
    'test5.jpg': ['fried rice'],
    'test6.webp': ['hoppers'],
    'test7.PNG': ['rice', 'pol sambol', 'dhal curry', 'beetroot curry', 'gotukola mallum'],
    'test8.jpg': ['rice', 'fish curry', 'carrot curry', 'gotukola mallum'],
    'test9.jpg': ['donut'],
    'test10.PNG': ['roll'],
    'test11.webp': ['rice', 'gotukola mallum', 'chicken curry', 'sprats', 'beans curry', 'beetroot curry'],
    'test12.jpg': ['gotukola mallum', 'red rice', 'pol sambol', 'dhal curry', 'fish curry']
}

# Specific foods and obsolete categories
specific_foods = [
    'donut', 'eclair', 'cake', 'brownie', 'pastry', 'cutlets', 'roll',
    'cream bun', 'crocodile bun', 'fish bun', 'sausage hotdog',
    'kiribath', 'kottu', 'hoppers', 'string hoppers', 'wade', 'pittu',
    'coconut roti', 'watalappam', 'lunu sambol', 'pol sambol', 'papadam',
    'gotukola mallum', 'moringa curry'
]

obsolete = ['fried filled', 'baked filled', 'baked sweet bun', 'sweets']

def ensemble_detect(image_path, confidence=0.25):
    r1 = v21(image_path, conf=confidence)[0]
    r2 = v24(image_path, conf=confidence)[0]
    r3 = v25(image_path, conf=confidence)[0]

    v21_foods = [r1.names[int(b.cls[0])].lower() for b in r1.boxes]
    v24_foods = [r2.names[int(b.cls[0])].lower() for b in r2.boxes]
    v25_foods = [r3.names[int(b.cls[0])].lower() for b in r3.boxes]

    v21_unique = list(set(v21_foods))
    v24_unique = list(set(v24_foods))
    v25_unique = list(set(v25_foods))

    final = []

    for f in v25_unique:
        if f in specific_foods:
            final.append(f)

    all_foods = v21_unique + v24_unique + v25_unique
    all_foods = [f for f in all_foods if f not in obsolete]
    all_foods = [f for f in all_foods if f not in final]

    vote_count = Counter()
    for f in set(all_foods):
        votes = (f in v21_unique) + (f in v24_unique) + (f in v25_unique)
        vote_count[f] = votes

    for f, votes in vote_count.items():
        if votes >= 2:
            final.append(f)

    return list(set(final))

# Upload test images
print("\n📤 Upload test images:")
uploaded = files.upload()

results = []

for filename in uploaded.keys():
    save_path = f"test_{filename}"
    with open(save_path, 'wb') as f:
        f.write(uploaded[filename])

    # Get predictions
    r1 = v21(save_path)[0]
    r2 = v24(save_path)[0]
    r3 = v25(save_path)[0]

    v21_pred = [r1.names[int(b.cls[0])] for b in r1.boxes]
    v24_pred = [r2.names[int(b.cls[0])] for b in r2.boxes]
    v25_pred = [r3.names[int(b.cls[0])] for b in r3.boxes]

    ensemble_result = ensemble_detect(save_path)

    # Get ground truth
    gt = ground_truth.get(filename, [])

    # Calculate status
    gt_set = set(gt)
    ensemble_set = set(ensemble_result)

    correct = gt_set & ensemble_set
    missed = gt_set - ensemble_set
    false_pos = ensemble_set - gt_set

    status = "✅ Full" if not missed and not false_pos else "⚠️ Partial"

    results.append({
        'Image': filename,
        'Ground Truth': ', '.join(gt),
        'V21': v21_pred,
        'V24': v24_pred,
        'V25': v25_pred,
        'Ensemble': ensemble_result,
        'Status': status,
        'Missed': list(missed),
        'False Positives': list(false_pos)
    })

# Display results
print("\n" + "="*100)
print("ENSEMBLE TESTING RESULTS WITH GROUND TRUTH")
print("="*100)

for r in results:
    print(f"\n📸 {r['Image']}")
    print(f"   Ground Truth: {r['Ground Truth']}")
    print(f"   V21: {r['V21']}")
    print(f"   V24: {r['V24']}")
    print(f"   V25: {r['V25']}")
    print(f"   Ensemble: {r['Ensemble']}")
    print(f"   Status: {r['Status']}")
    if r['Missed']:
        print(f"   ❌ Missed: {r['Missed']}")
    if r['False Positives']:
        print(f"   ⚠️ False Positives: {r['False Positives']}")

# Summary
print("\n" + "="*100)
print("PERFORMANCE SUMMARY")
print("="*100)

total = len(results)
full = sum(1 for r in results if r['Status'] == '✅ Full')
print(f"Total Images: {total}")
print(f"Fully Correct: {full}")
print(f"Partially Correct: {total - full}")
print(f"Accuracy: {full/total*100:.1f}%")

# Save to CSV
df = pd.DataFrame(results)
df.to_csv('ensemble_test_results_with_gt.csv', index=False)
print("\n✅ Results saved to 'ensemble_test_results_with_gt.csv'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.6 MB/s eta 0:00:00
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading models...
✅ Models loaded!

📤 Upload test images:


Saving test1.jpg to test1.jpg
Saving test2.jpg to test2.jpg
Saving test3.jpg to test3.jpg
Saving test4.jpg to test4.jpg
Saving test5.jpg to test5.jpg
Saving test6.webp to test6.webp
Saving test7.PNG to test7.PNG
Saving test8.jpg to test8.jpg
Saving test9.jpg to test9.jpg
Saving test10.PNG to test10.PNG
Saving test11.webp to test11.webp
Saving test12.jpg to test12.jpg

image 1/1 /content/test_test1.jpg: 384x640 1 Cabbage Curry, 1 Carrot, 576.0ms
Speed: 14.5ms preprocess, 576.0ms inference, 44.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/test_test1.jpg: 384x640 1 Egg curry, 1 Kottu, 579.3ms
Speed: 3.7ms preprocess, 579.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/test_test1.jpg: 384x640 1 Kottu, 1 Potato Curry, 563.0ms
Speed: 3.8ms preprocess, 563.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/test_test1.jpg: 384x640 1 Cabbage Curry, 1 Carrot, 556.4ms
Speed: 3.7ms preprocess, 